In [ ]:
# If needed, install the notebook dependencies in the active kernel:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("insurance2 (1).csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Run this notebook from the repository root."
    )

df = pd.read_csv(DATA_PATH)
target_column = "insuranceclaim"
if target_column not in df.columns:
    raise ValueError(f"Expected target column {target_column!r}; found {df.columns.tolist()}")

print("First five records")
display(df.head())
print("Last five records")
display(df.tail())
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
print("\nData types")
display(df.dtypes.rename("dtype").to_frame())
print("Missing values by column")
display(df.isna().sum().rename("missing_count").to_frame())

## Preprocessing and holdout split

The test set is split off before fitting any imputer, scaler, or encoder. Those transformations live inside a scikit-learn pipeline, so their statistics and category vocabulary are learned from training data only. This avoids leaking information about the test set into training.

In [ ]:
numeric_columns = ["age", "bmi", "children", "charges"]
categorical_columns = ["sex", "smoker", "region"]
required_columns = numeric_columns + categorical_columns + [target_column]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

X = df[numeric_columns + categorical_columns].copy()
y = df[target_column].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_columns),
    ("categorical", categorical_preprocessor, categorical_columns),
])

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

## Train and evaluate a classifier

A class-weighted logistic regression is used as an interpretable baseline. The threshold is left at its standard 0.5 for this first evaluation; any threshold adjustment should be selected with validation data and a clearly stated business cost, never by tuning on the final test set.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]
metrics = pd.DataFrame([{
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_claim": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "recall_claim": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "f1_claim": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_probability),
}]).T.rename(columns={0: "test_score"})
display(metrics.round(3))

print("Confusion matrix (rows = actual, columns = predicted; labels 0, 1):")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["actual_no_claim", "actual_claim"],
    columns=["predicted_no_claim", "predicted_claim"],
))
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))